![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Custom Data Research

This notebook defines the Bitstamp custom data type and builds the daily Bitcoin price series.

### Set Up QuantBook

Create the QuantBook for the custom data history request.

In [ ]:
qb = QuantBook()
qb.set_start_date(2020, 12, 31)

### Add Custom Data

Define a [custom securities](https://www.quantconnect.com/docs/v2/writing-algorithms/importing-data/streaming-data/custom-securities) type and subscribe to it.

In [ ]:
class Bitstamp(PythonData):

    def get_source(self, config: SubscriptionDataConfig, date: datetime, is_live_mode: bool) -> SubscriptionDataSource:
        return SubscriptionDataSource(
            "https://raw.githubusercontent.com/QuantConnect/Documentation/master/Resources/datasets/custom-data/bitstampusd.csv",
            SubscriptionTransportMedium.REMOTE_FILE
        )

    def reader(self, config: SubscriptionDataConfig, line: str, date: datetime, is_live_mode: bool) -> BaseData:
        if not line.strip() or not line[0].isdigit():
            return None
        data = line.split(',')
        coin = Bitstamp()
        coin.symbol = config.symbol
        coin.value = float(data[4])
        if coin.value == 0:
            return None
        coin.time = datetime.strptime(data[0], "%Y-%m-%d")
        coin.end_time = coin.time + timedelta(1)
        coin["Open"] = float(data[1])
        coin["High"] = float(data[2])
        coin["Low"] = float(data[3])
        coin["Close"] = coin.value
        coin["VolumeBTC"] = float(data[5])
        coin["VolumeUSD"] = float(data[6])
        coin["WeightedPrice"] = float(data[7])
        return coin

In [ ]:
btc = qb.add_data(Bitstamp, "BTC")

### Build Time Series

Request daily history for the custom type and build the close-price series.

In [ ]:
history = qb.history(btc.symbol, 200, Resolution.DAILY)
history

In [ ]:
# Pull the parsed close price into a single series.
closes = history["close"]
closes